# Tutorial 01: Discover Pipeline

This notebook walks through the first two layers of pynpxpipe:
**Core infrastructure** (Session, config, logging) and **IO + Discover stage**.

By the end you will have:
- A fully initialized `Session` object
- `session.probes` populated with your recording's probe metadata
- Five visualizations of your data
- An understanding of how checkpoints enable resume-on-failure

**Prerequisites:** A SpikeGLX recording folder and a `.bhv2` behavioral file on disk.

---

In [ ]:
# === 用户配置区（修改这里）===========================================
# DATA_DIR: 根目录，需包含 SpikeGLX gate 文件夹（*_g0/ 等）和 .bhv2 文件
# OUTPUT_DIR: 处理结果输出目录（不存在会自动创建）
# SUBJECT_YAML: monkeys/*.yaml 中对应实验动物的配置文件

from pathlib import Path

DATA_DIR     = Path(r"C:\your\recording\root")   # <-- 修改这里
OUTPUT_DIR   = Path(r"C:\your\output")            # <-- 修改这里
SUBJECT_YAML = Path(r".\monkeys\YourMonkey.yaml") # <-- 修改这里
# =====================================================================

# Validate — friendly error messages instead of raw Python tracebacks
issues = []
if not DATA_DIR.exists():
    issues.append(f"❌ DATA_DIR not found: {DATA_DIR}")
if not SUBJECT_YAML.exists():
    issues.append(f"❌ SUBJECT_YAML not found: {SUBJECT_YAML}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if issues:
    for msg in issues:
        print(msg)
    raise SystemExit("请修改上方配置区中的路径，再重新运行此 cell。")

print("✅ 路径验证通过")
print(f"   DATA_DIR     = {DATA_DIR}")
print(f"   OUTPUT_DIR   = {OUTPUT_DIR}")
print(f"   SUBJECT_YAML = {SUBJECT_YAML}")

## Section 01: Session Setup

`Session` is pynpxpipe's central state object — a plain Python dataclass passed
through every stage. It holds paths, subject metadata, the probe list (populated
by DiscoverStage), and per-stage checkpoint status.

`SessionManager.from_data_dir()` auto-discovers the SpikeGLX gate folder
(`*_g0/` etc.) and the `.bhv2` file within `DATA_DIR`, then creates the output
directory structure and initializes logging.

**Design note:** `Session` has zero UI dependencies — no `click`, no `print`.
This means the same `Session` object works identically whether called from CLI,
this notebook, or a future GUI frontend.

In [ ]:
from pynpxpipe.core.config import load_subject_config
from pynpxpipe.core.session import Session, SessionManager
from pynpxpipe.core.logging import setup_logging

# 1. Load subject metadata from YAML
subject = load_subject_config(SUBJECT_YAML)
print(f"Subject loaded: {subject.subject_id} ({subject.species}, {subject.sex}, {subject.age})")

# 2. Create session — auto-discovers gate dir and .bhv2 inside DATA_DIR
#    Also creates: OUTPUT_DIR/checkpoints/, OUTPUT_DIR/logs/
session = SessionManager.from_data_dir(
    data_dir=DATA_DIR,
    subject=subject,
    output_dir=OUTPUT_DIR,
)
print(f"\nSession created:")
print(f"  session_dir : {session.session_dir}")
print(f"  output_dir  : {session.output_dir}")
print(f"  bhv_file    : {session.bhv_file}")
print(f"  probes      : {len(session.probes)} (populated after DiscoverStage)")

# 3. Setup structured logging (writes to OUTPUT_DIR/logs/*.log)
#    After this call, all stage output goes to the log file as JSON Lines.
setup_logging(session.log_path)
print(f"\nLogging to: {session.log_path}")

## Section 02: SpikeGLX Data Discovery (IO Layer)

`SpikeGLXDiscovery` scans the session directory and returns metadata for each
probe — **without loading any `.bin` data**. The `.bin` files can be 400-500 GB;
lazy loading via SpikeInterface means only the metadata is read until you
explicitly request traces.

`SpikeGLXLoader.load_nidq()` returns a lazy `BaseRecording`. Only when you call
`get_traces()` does data actually move from disk to RAM — and only the requested
chunk.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pynpxpipe.io.spikeglx import SpikeGLXDiscovery, SpikeGLXLoader

# --- 1. Discover all probes (reads .meta, not .bin) ---
discovery = SpikeGLXDiscovery(session.session_dir)
probes = discovery.discover_probes()
print(f"Found {len(probes)} probe(s):")
for p in probes:
    print(f"  {p.probe_id}: {p.probe_type}, SN={p.serial_number}, "
          f"{p.n_channels} ch @ {p.sample_rate/1000:.1f} kHz")

# --- 2. Discover NIDQ ---
nidq_bin, nidq_meta = discovery.discover_nidq()
print(f"\nNIDQ: {nidq_bin.name}")

# --- 3. Lazy-load NIDQ (no data read yet) ---
nidq_rec = SpikeGLXLoader.load_nidq(nidq_bin, nidq_meta)
fs        = nidq_rec.get_sampling_frequency()
ch_ids    = nidq_rec.get_channel_ids()
print(f"NIDQ: {len(ch_ids)} channels @ {fs:.0f} Hz")
print(f"Channels: {list(ch_ids)}")

# --- 4. Visualization A: Raw NIDQ — first 5 seconds ---
n_samples = int(5 * fs)
traces    = nidq_rec.get_traces(start_frame=0, end_frame=n_samples)  # shape: (n_samples, n_ch)
t         = np.arange(n_samples) / fs

fig, axes = plt.subplots(len(ch_ids), 1, figsize=(14, 2 * len(ch_ids)), sharex=True)
if len(ch_ids) == 1:
    axes = [axes]

for ax, ch_name, col in zip(axes, ch_ids, traces.T):
    ax.plot(t, col, lw=0.6)
    ax.set_ylabel(ch_name, fontsize=8, rotation=0, labelpad=40, va="center")
    ax.spines[["top", "right"]].set_visible(False)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Raw NIDQ Signal — First 5 seconds", y=1.01)
plt.tight_layout()
plt.show()
print("\n💡 注意：traces 仅从磁盘读取了前 5 秒的数据，.bin 文件其余内容未加载。")

## Section 03: DiscoverStage

`DiscoverStage` wraps the IO-layer discovery in the pipeline's stage protocol:
it checks for a completed checkpoint first (skip if done), runs discovery,
validates probe + NIDQ + BHV2 integrity, populates `session.probes`, writes
`session_info.json`, and saves a checkpoint.

Every stage follows this same contract: **checkpoint-first, fail-fast, structured log.**

In [ ]:
import json
from pynpxpipe.stages.discover import DiscoverStage

stage = DiscoverStage(session)
stage.run()

print(f"session.probes populated: {len(session.probes)} probe(s)")
for p in session.probes:
    print(f"  {p.probe_id}: {p.n_channels} channels, {p.sample_rate/1000:.1f} kHz")

# Read and display session_info.json
session_info_path = session.output_dir / "session_info.json"
session_info = json.loads(session_info_path.read_text(encoding="utf-8"))
print("\nsession_info.json:")
print(json.dumps(session_info, indent=2))